# Task 2: Hourly Pollutant Forecasting — Production Pipeline v3

**Model**: LightGBM ensemble with log1p target transform, Fourier features, anchor lags, target encoding, cross-station spatial features, and conformalized quantile regression (CQR) for calibrated 90% prediction intervals.

**Ensemble**: Optimized weighted average of LightGBM + Ridge (Fourier) + Seasonal Naive.

**Validation**: Walk-forward CV with 3 folds × 720h test windows.

In [ ]:
import sys, os
sys.path.insert(0, '..')
os.makedirs("../outputs", exist_ok=True)

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data.loader import load_series
from src.forecasting.train import (
    train_forecast_pipeline, predict_with_pipeline,
    seasonal_naive_predict, walk_forward_cv
)
from src.forecasting.evaluate import evaluate_predictions, evaluate_intervals
from src.utils.constants import FORECAST_TARGETS

print(f"Targets: {len(FORECAST_TARGETS)}")
for t in FORECAST_TARGETS:
    print(f"  Station {t['station_code']}/{t['item_name']} → {t['start']} to {t['end']}")

In [ ]:
results = []
all_predictions = {}

for target in FORECAST_TARGETS:
    sc, ic, name = target["station_code"], target["item_code"], target["item_name"]
    pred_start, pred_end = target["start"], target["end"]

    print(f"\n{'='*70}")
    print(f"Station {sc} / {name} | Predict: {pred_start} to {pred_end}")
    print(f"{'='*70}")

    # Load and prepare
    raw = load_series(sc, ic, normal_only=True, end_before=pred_start)
    ts = raw["clean_value"].copy()
    full_idx = pd.date_range(ts.index.min(), ts.index.max(), freq="h")
    ts = ts.reindex(full_idx).ffill().bfill()
    overall_std = ts.std()

    print(f"Training: {len(ts)} hours | Std: {overall_std:.5f}")

    # --- Walk-forward CV ---
    folds = walk_forward_cv(ts, n_folds=3, test_size=720, min_train_size=8760)
    cv_metrics = {"naive": [], "lgbm": [], "ensemble": []}
    cv_intervals = []

    for fold in folds:
        ft = ts.iloc[:fold["train_end"]]
        fv = ts.iloc[fold["test_start"]:fold["test_end"]]

        pipe = train_forecast_pipeline(ft, fv, station_code=sc, item_code=ic)
        preds = predict_with_pipeline(pipe, fv.index)

        for mn, col in [("naive", "naive"), ("lgbm", "lgbm"), ("ensemble", "ensemble")]:
            m = evaluate_predictions(fv, preds[col])
            m["nrmse_overall"] = m["rmse"] / overall_std
            cv_metrics[mn].append(m)

        iv = evaluate_intervals(fv.values, preds["q05"].values, preds["q95"].values)
        cv_intervals.append(iv)

    for mn in cv_metrics:
        avg = {k: np.mean([m[k] for m in cv_metrics[mn]]) for k in cv_metrics[mn][0]}
        print(f"  {mn:>8}: RMSE={avg['rmse']:.5f}, nRMSE={avg['nrmse_overall']:.3f}, "
              f"R²={avg['r2']:.3f}, MAE={avg['mae']:.5f}")

    avg_iv = {k: np.mean([iv[k] for iv in cv_intervals]) for k in cv_intervals[0]}
    print(f"  Intervals: coverage={avg_iv['empirical_coverage']:.3f} (target 0.90), "
          f"width={avg_iv['avg_interval_width']:.5f}")

    # --- Final prediction ---
    val_start = ts.index.max() - pd.DateOffset(months=1) + pd.Timedelta(hours=1)
    train_final = ts.loc[:val_start - pd.Timedelta(hours=1)]
    val_final = ts.loc[val_start:]

    final_pipe = train_forecast_pipeline(train_final, val_final, station_code=sc, item_code=ic)
    full_pipe = train_forecast_pipeline(ts, station_code=sc, item_code=ic)
    full_pipe["weights"] = final_pipe["weights"]
    full_pipe["cqr_correction"] = final_pipe["cqr_correction"]

    pred_index = pd.date_range(pred_start, pred_end, freq="h")
    final_preds = predict_with_pipeline(full_pipe, pred_index)

    print(f"\n  Weights: { {k: round(v, 3) for k, v in full_pipe['weights'].items()} }")
    print(f"  CQR correction: {full_pipe['cqr_correction']:.5f}")
    print(f"  Predictions: {len(final_preds)} hours, "
          f"range=[{final_preds['ensemble'].min():.5f}, {final_preds['ensemble'].max():.5f}]")

    avg_ens = {k: np.mean([m[k] for m in cv_metrics["ensemble"]]) for k in cv_metrics["ensemble"][0]}
    avg_nav = {k: np.mean([m[k] for m in cv_metrics["naive"]]) for k in cv_metrics["naive"][0]}

    results.append({
        "station": sc, "pollutant": name,
        "cv_naive_rmse": round(avg_nav["rmse"], 5),
        "cv_ensemble_rmse": round(avg_ens["rmse"], 5),
        "cv_nrmse": round(avg_ens["nrmse_overall"], 3),
        "cv_r2": round(avg_ens["r2"], 3),
        "improvement_%": round((1 - avg_ens["rmse"] / avg_nav["rmse"]) * 100, 1),
        "interval_coverage": round(avg_iv["empirical_coverage"], 3),
    })

    all_predictions[f"{sc}_{name}"] = {
        "final": final_preds,
        "cv_last_fold_preds": preds,
        "cv_last_fold_actual": fv,
    }

print("\n\nAll targets processed.")

In [ ]:
results_df = pd.DataFrame(results)
print("Walk-Forward CV Results (3 folds × 720h):\n")
print(results_df.to_string(index=False))

In [ ]:
# Validation plot: last CV fold with calibrated intervals
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

for i, (key, data) in enumerate(all_predictions.items()):
    ax = axes[i]
    actual = data["cv_last_fold_actual"]
    preds = data["cv_last_fold_preds"]
    n = min(168, len(actual))
    t = actual.index[:n]

    ax.fill_between(t, preds["q05"].iloc[:n], preds["q95"].iloc[:n],
                     alpha=0.2, color="C1", label="90% PI (CQR)")
    ax.plot(t, actual.iloc[:n], lw=1, alpha=0.8, label="Actual", color="C0")
    ax.plot(t, preds["ensemble"].iloc[:n], lw=1, alpha=0.8, label="Ensemble", color="C1")
    ax.set_title(key.replace("_", " / "), fontsize=11)
    ax.legend(fontsize=7, loc="upper right")
    ax.tick_params(axis='x', rotation=30, labelsize=7)

plt.suptitle("CV Last Fold (7 days): Ensemble + CQR 90% Prediction Intervals", fontsize=13)
plt.tight_layout()
plt.savefig("../outputs/forecast_validation_v3.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Export final predictions with calibrated intervals
all_exports = []
for target in FORECAST_TARGETS:
    key = f"{target['station_code']}_{target['item_name']}"
    preds = all_predictions[key]["final"]
    df = pd.DataFrame({
        "measurement_datetime": preds.index,
        "station_code": target["station_code"],
        "item_code": target["item_code"],
        "item_name": target["item_name"],
        "predicted_value": preds["ensemble"].values,
        "predicted_lower_90": preds["q05"].values,
        "predicted_upper_90": preds["q95"].values,
    })
    all_exports.append(df)

export_df = pd.concat(all_exports, ignore_index=True)
export_df.to_csv("../outputs/forecast_predictions.csv", index=False)
print(f"Exported {len(export_df)} predictions with CQR-calibrated 90% intervals")
print(export_df.groupby(["station_code", "item_name"]).agg(
    count=("predicted_value", "count"),
    mean=("predicted_value", lambda x: round(x.mean(), 5)),
    min=("predicted_value", lambda x: round(x.min(), 5)),
    max=("predicted_value", lambda x: round(x.max(), 5)),
).to_string())